# City2TABULA Validation Notebook

This notebook validates the calculations performed by the City2TABULA pipeline by comparing calculated building attributes against source thematic data from CityGML/CityJSON datasets.

## Validation Strategy

1. **Building-Level Attributes**: Height, footprint area, aggregated surface areas
2. **Surface-Level Attributes**: Individual surface area, tilt (roof only), azimuth (roof only)

The validation uses a configuration-driven approach where source property names are mapped to City2TABULA calculated columns via YAML configuration files.

In [ ]:
import sys, os
from datetime import datetime
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

from modules.config  import load_config, print_config_summary, \
                            get_building_attribute_mapping, get_surface_attribute_mapping
from modules.db      import get_db_engine, close_db_engine
from modules.utils   import load_city2tabula_data, \
                            load_thematic_building_data, load_thematic_surface_data
from modules.validators import (validate_building_attributes, validate_surface_attributes,
                                 get_validation_summary, export_problematic_surfaces,
                                 analyze_geometry_quality)
from modules.plots   import (plot_comparison_scatter, plot_error_distribution,
                              plot_percent_error_distribution, plot_multi_attribute_comparison,
                              plot_height_neighbour_split)


## Stage 0: Setup

Load configuration from `configs/config_{COUNTRY}.yaml` and connect to the database.

In [ ]:
# Add validation directory to path so modules can be imported
notebook_dir = os.path.abspath('')
if notebook_dir not in sys.path:
    sys.path.insert(0, notebook_dir)

country = os.getenv('COUNTRY', '').lower()
db_name = os.getenv('DB_NAME', 'unknown')

config_path = os.path.join('configs', f'config_{country}.yaml')
config = load_config(config_path)
print_config_summary(config)

output_dir = 'outputs'
os.makedirs(output_dir, exist_ok=True)

fig_format = 'png'   # 'png' | 'svg' | 'pdf' | 'ipe'

db_engine = get_db_engine(config)
print(f"Connected to: {db_name}")


## Stage 1: Load Data

Load City2TABULA calculated features from the database.

In [ ]:
bf_df, sf_df = load_city2tabula_data(db_engine, config)

print(f"\nBuildings : {len(bf_df):,}")
print(f"Surfaces  : {len(sf_df):,}")
display(bf_df.head(2))
display(sf_df.head(2))


## Stage 2: Validate Attributes

Compare City2TABULA calculated values against source thematic data from CityDB.

In [ ]:
# ── Attribute mappings ────────────────────────────────────────────────────────
building_attr_map = get_building_attribute_mapping(config)
roof_attr_map     = get_surface_attribute_mapping(config, 'roof')
wall_attr_map     = get_surface_attribute_mapping(config, 'wall')
floor_attr_map    = get_surface_attribute_mapping(config, 'floor')

building_ids  = bf_df['building_feature_id'].tolist()
roof_ids      = sf_df[sf_df['classname'] == 'RoofSurface']['surface_feature_id'].tolist()
wall_ids      = sf_df[sf_df['classname'] == 'WallSurface']['surface_feature_id'].tolist()
floor_ids     = sf_df[sf_df['classname'] == 'GroundSurface']['surface_feature_id'].tolist()

# ── Load thematic data ────────────────────────────────────────────────────────
print("Loading thematic data...")
building_thematic_df = load_thematic_building_data(db_engine, config, building_ids, building_attr_map)
roof_thematic_df     = load_thematic_surface_data(db_engine, config, roof_ids,  roof_attr_map,  'RoofSurface')
wall_thematic_df     = load_thematic_surface_data(db_engine, config, wall_ids,  wall_attr_map,  'WallSurface')
floor_thematic_df    = load_thematic_surface_data(db_engine, config, floor_ids, floor_attr_map, 'GroundSurface')

# ── Run validation ────────────────────────────────────────────────────────────
print("\nRunning validation...")
building_validation_df = validate_building_attributes(bf_df, building_thematic_df, building_attr_map)
roof_validation_df     = validate_surface_attributes(sf_df, roof_thematic_df,     roof_attr_map,     'RoofSurface')
wall_validation_df     = validate_surface_attributes(sf_df, wall_thematic_df,     wall_attr_map,     'WallSurface')
floor_validation_df    = validate_surface_attributes(sf_df, floor_thematic_df,    floor_attr_map,    'GroundSurface')

# ── Summaries ─────────────────────────────────────────────────────────────────
building_summary = get_validation_summary(building_validation_df)
roof_summary     = get_validation_summary(roof_validation_df)
wall_summary     = get_validation_summary(wall_validation_df)
floor_summary    = get_validation_summary(floor_validation_df)

for label, df in [('Building', building_summary), ('Roof', roof_summary),
                   ('Wall', wall_summary), ('Floor/Ground', floor_summary)]:
    if not df.empty:
        print(f"\n── {label} ──")
        display(df)


In [ ]:
timestamp   = datetime.now().strftime('%Y%m%d_%H%M%S')
results_dir = os.path.join(output_dir, config['dataset']['country'], db_name,
                            f'validation_{timestamp}')
os.makedirs(results_dir, exist_ok=True)
plots_dir   = os.path.join(results_dir, 'plots')
os.makedirs(plots_dir, exist_ok=True)

pairs = [
    ('building', building_validation_df, building_summary),
    ('roof',     roof_validation_df,     roof_summary),
    ('wall',     wall_validation_df,     wall_summary),
    ('floor',    floor_validation_df,    floor_summary),
]

for name, val_df, summ_df in pairs:
    if val_df.empty:
        continue
    val_df.to_csv(os.path.join(results_dir, f'{name}_validation.csv'), index=False)
    summ_df.to_csv(os.path.join(results_dir, f'{name}_summary.csv'),    index=False)

print(f"Results saved to: {results_dir}")


## Stage 2.5: Export Problematic Surfaces

Export surfaces above the error threshold for QGIS inspection.

In [ ]:
error_threshold = 10.0

for name, val_df in [('roofs',  roof_validation_df),
                      ('walls',  wall_validation_df),
                      ('floors', floor_validation_df)]:
    if not val_df.empty:
        export_problematic_surfaces(val_df,
                                     os.path.join(results_dir, f'problematic_{name}.csv'),
                                     error_threshold)


## Stage 3: Validation Plots

Scatter plots and error distributions for all validated attributes.

In [ ]:
for name, val_df, prefix in [('building', building_validation_df, 'Building'),
                               ('roof',     roof_validation_df,     'Roof'),
                               ('wall',     wall_validation_df,     'Wall'),
                               ('floor',    floor_validation_df,    'Floor')]:
    if val_df.empty:
        continue

    plot_multi_attribute_comparison(
        val_df,
        save_path=os.path.join(plots_dir, f'{name}_multi_comparison.{fig_format}'),
        title_prefix=prefix, fig_format=fig_format)

    for attr in val_df['attribute_name'].unique():
        plot_comparison_scatter(val_df, attr,
            save_path=os.path.join(plots_dir, f'{name}_{attr}_scatter.{fig_format}'))
        plot_error_distribution(val_df, attr,
            save_path=os.path.join(plots_dir, f'{name}_{attr}_error_dist.{fig_format}'))
        plot_percent_error_distribution(val_df, attr,
            save_path=os.path.join(plots_dir, f'{name}_{attr}_pct_error.{fig_format}'))

print(f"Plots saved to: {plots_dir}")


## Stage 3.5: Height Validation — Standalone vs Attached Buildings

Split height accuracy by `has_attached_neighbour` flag.  Buildings whose footprints
overlap or touch neighbour geometries can receive misassigned child surfaces, inflating
or deflating derived min/max heights.  This plot isolates that effect.

Run `python flag_attached_buildings.py` before this cell to populate the flag.


In [ ]:
if not building_validation_df.empty and 'has_attached_neighbour' in building_validation_df.columns:
    height_attrs = [a for a in ['min_height', 'max_height']
                    if a in building_validation_df['attribute_name'].unique()]

    for attr in height_attrs:
        sub = building_validation_df[building_validation_df['attribute_name'] == attr]
        print(f"\n  {attr}:")
        for flag, label in [(False, 'Standalone'), (True, 'Attached')]:
            grp = sub[sub['has_attached_neighbour'] == flag]
            if grp.empty:
                continue
            rmse      = float(np.sqrt((grp['difference']**2).mean()))
            mean_diff = float(grp['difference'].mean())
            print(f"    {label:12s}  n={len(grp):6d}  RMSE={rmse:.3f} m  mean={mean_diff:+.3f} m")

    plot_height_neighbour_split(
        building_validation_df,
        attributes=height_attrs,
        save_path=os.path.join(plots_dir, f'building_height_neighbour_split.{fig_format}'),
        fig_format=fig_format)
else:
    print("Skipping: no building validation data or has_attached_neighbour flag not available.\n"
          "Run  python flag_attached_buildings.py  first.")


## Stage 4: Summary

Final statistics across all validated attributes.

In [ ]:
total = sum(len(df) for df in [building_validation_df, roof_validation_df,
                                 wall_validation_df,    floor_validation_df]
             if not df.empty)

print(f"Total validation comparisons : {total:,}")
print(f"Results directory            : {results_dir}")

for label, df in [('Building', building_summary), ('Roof', roof_summary),
                   ('Wall', wall_summary), ('Floor/Ground', floor_summary)]:
    if not df.empty:
        print(f"\n── {label} ──")
        display(df)


In [ ]:
close_db_engine(db_engine)
print("Database connection closed.")
